# Lesson 10

## Video 2 — Defining Your First Tool

In [114]:
# Import the tool definition and Python function
from claude_assist.tools import CALCULATOR_TOOL, calculate

print(CALCULATOR_TOOL)

{'name': 'calculate', 'description': 'Perform basic arithmetic using two numbers.', 'input_schema': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}, 'operation': {'type': 'string', 'enum': ['add', 'subtract', 'multiply', 'divide']}}, 'required': ['a', 'b', 'operation']}}


In [115]:
# Test the Python function directly
print(calculate(18, 27, "multiply"))

486


## Video 3 — Detecting and Running a Tool

In [116]:
# Set up the Claude client
from dotenv import load_dotenv
import anthropic

from claude_assist import config

load_dotenv()
client = anthropic.Anthropic()

In [117]:
# Ask a question that should trigger the calculator tool
messages = [
    {
        "role": "user",
        "content": "What is 137 multiplied by 48? Use the calculator tool."
    }
]

resp = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=[CALCULATOR_TOOL],
    messages=messages
)

print("Stop reason:", resp.stop_reason)
print(resp.content)

Stop reason: tool_use
[ToolUseBlock(id='toolu_01NvAM2yi83VXZL8hDH6tzDF', caller=DirectCaller(type='direct'), input={'a': 137, 'b': 48, 'operation': 'multiply'}, name='calculate', type='tool_use', toolset_name=None)]


In [118]:
# Find the tool request
for block in resp.content:
    if block.type == "tool_use":
        print("Tool:", block.name)
        print("Inputs:", block.input)
        print("ID:", block.id)

Tool: calculate
Inputs: {'a': 137, 'b': 48, 'operation': 'multiply'}
ID: toolu_01NvAM2yi83VXZL8hDH6tzDF


In [120]:
# Run the requested calculator function
for block in resp.content:
    if block.type == "tool_use" and block.name == "calculate":
        args = block.input

        result = calculate(
            args["a"],
            args["b"],
            args["operation"]
        )

        tool_block = block

        print("Result:", result)

Result: 6576


## Video 4 — Completing the Full Tool-Use Cycle

In [124]:
# Keep Claude's tool request in the message history
messages.append({
    "role": "assistant",
    "content": resp.content
})

# Add the calculator result as a tool_result
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,
        "content": str(result)
    }]
})

In [132]:
# Call Claude again so it can use the tool result
final = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=[CALCULATOR_TOOL],
    messages=messages
)

reply = "".join(block.text for block in final.content if block.type == "text")
print(reply)

137 multiplied by 48 is **6,576**.

I ran the calculation three times by mistake, and each run gave the same result.


The full cycle is now complete:

1. User asks a question
2. Claude requests the calculator
3. Python runs the calculator
4. We return the `tool_result`
5. Claude gives the final answer